## To run simulations iteratively

In [ ]:
using NPZ,Interpolations,Plots,Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, ColorSchemes, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, Wandb, PyCall,Logging, GLM, PyPlot

In [ ]:
## Code to iteratively run simulations

using NPZ,Interpolations,Plots,Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, ColorSchemes, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, Wandb, PyCall,Logging, GLM, PyPlot

folder_path = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/utils/functions_run_simulations"
for file in readdir(folder_path)
    if endswith(file, ".jl")
        include(joinpath(folder_path, file))
    end
end

# List of days to process
days=[ 1, 3] # 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

# Load static neuron properties
JLD2.@load "Data_MN_Model/Data_final/Subjects_optim_process/VthCa_TA_subject1_gamma1.jld2" VthCa_newTA
nr_s = length(VthCa_newTA)

# Precompute neuron parameters (as before)
rheobase_S = LinRange(3.5, 6.5, nr_s) .+ rand(Normal(0.0, 0.05), nr_s)
rs_S = LinRange(38.75, 41.25, nr_s) ./ 10000
ls_S = LinRange(77.5, 82.5, nr_s) ./ 10000
rms_S = LinRange(1.15, 1.05, nr_s) .* 1000
rd_S = LinRange(20.75, 31.25, nr_s) ./ 10000
ld_S = LinRange(5500., 6800., nr_s) ./ 10000
rmd_S = LinRange(14.4, 10.7, nr_s) .* 1000

max_gna = 30.0
max_gkf_S = ones(nr_s) * 4
max_gks_S = LinRange(16, 25, nr_s)
max_gca_S = LinRange(0.038, 0.029, nr_s)

αm = ones(nr_s) * 22.0
βm = ones(nr_s) * 13.0
αh_S = ones(nr_s) * 0.5
βh_S = ones(nr_s) * 4
αn_S = ones(nr_s) * 1.5
βn_S = ones(nr_s) * 0.1
αq_S = ones(nr_s) * 1.5
βq_S = LinRange(0.025, 0.038, nr_s)
αp_D = 0.008
βp_D = LinRange(0.014, 0.016, nr_s)

Vthca_S = VthCa_newTA[1, :]
Apeak_S = LinRange(0.103, 0.123, nr_s)
tpeak_S = LinRange(110., 100., nr_s)
tetanic_S = LinRange(0.393, 0.491, nr_s)

γh = ones(nr_s)
IF = LinRange(1., 1., nr_s)

EL = 0; ENa = 120; ECa = 140; EK = -10; Ee = 70; Ei = -16
Ge = 0.6; Gi = 0.5
Vd0 = 0.; Vs0 = 0.;
rp = 5.0
pd=1; 

const step = 0.05
_int(num) = floor(Int, num)

# Precompute neuron properties
MN_list = []
for i in 1:nr_s
    gls = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6
    gld = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6
    gc = ((2/(((70*ld_S[i])/(π*rd_S[i]^2)) + ((70*ls_S[i])/(π*rs_S[i]^2)))))*1e6
    rn = 1 / (gls + ((gld*gc)/(gld+gc)))
    Vth = round(rn * rheobase_S[i], digits=2)
    cs = 2*π*ls_S[i]*rs_S[i]*1*1e3
    cd = 2*π*ld_S[i]*rd_S[i]*1*1e3
    gna = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3
    gkf = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3
    gks = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3
    gca = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3

    S_MN = Dict(:Cs => cs, :Cd => cd, :Irh => rheobase_S[i], :gc => gc,:gNa => gna, :gKs => gks, :gKf => gkf, :gCa => gca,  :gld => gld, :gls => gls, :αp => αp_D, :βp => βp_D[i], :αq => αq_S[i], :βq => βq_S[i], :αm => αm[i], :βm => βm[i],:αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:Vth => Vth, :VthCa => Vthca_S[i], :Rn => rn,:Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh => γh[i],"type" => "All Type")
    push!(MN_list, S_MN)
end
MNs = MN_list

for day in days # Loop over days

    day_str = lpad(day, 2, '0')  # Format day number with leading zero
    println("\nProcessing Day$day_str")

    # Load and resample force profile
    tr = npzread("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Scaled_force/S1/force_S1111_Day$(day_str).npy")
    sim_length = length(tr)
    interp = LinearInterpolation(1:sim_length, tr)
    resampled_indices = range(1, sim_length, length=Int(round(sim_length/2048))*1000)
    tr = interp(resampled_indices)
    sim_length = length(tr)
    amplitude = maximum(tr)

    # Simulation settings
    global tspan = 0:0.05:sim_length
    sim_time = _int(sim_length / 0.05)

    # Initialize state variables
    m=0.0; h=1.0; n=0.0; q=0.0; p=0.0; m0=0.0; h0=1.0; n0=0.0; q0=0.0; p0=0.0
    t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
    global pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s)

    global spike_arr = zeros((nr_s, sim_time))
    global vs_arr = zeros((nr_s, sim_time))
    global pt_arr = zeros((nr_s, sim_time))
    global mt_arr = zeros((nr_s, sim_time))
    global ht_arr = zeros((nr_s, sim_time))
    global nt_arr = zeros((nr_s, sim_time))
    global qt_arr = zeros((nr_s, sim_time))

    global commoninput = tr
    global inh_input = fill(0.00001 * amplitude, sim_length)
    global full_input = zeros((nr_s, sim_length))
    global independantnoise = zeros((nr_s, sim_length))

    # Run simulation
    @time spike_array, vs_array, pt_arr = run_sim()

    # Save results using JLD2.save
    output_file = "Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1/DR_rep_trial_$(day_str)_scaled11.jld2"
    JLD2.save(output_file, Dict("spike_array" => spike_array))
    println("Saved simulation results for Day$day_str to $output_file")
end


In [ ]:

folder_path = "./functions_run_simulations/"
for file in readdir(folder_path)
    if endswith(file, ".jl")
        include(joinpath(folder_path, file))
    end
end


force_dir = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Scaled_force/S7_alltrials/"
out_root  = "Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S7_alltrials"

JLD2.@load "Data_MN_Model/Data_final/Subjects_optim_process/VthCa_TA_subject1_gamma1.jld2" VthCa_newTA
nr_s = length(VthCa_newTA)

rheobase_S = LinRange(3.5, 6.5, nr_s) .+ rand(Normal(0.0, 0.05), nr_s)
rs_S = LinRange(38.75, 41.25, nr_s) ./ 10000
ls_S = LinRange(77.5, 82.5, nr_s) ./ 10000
rms_S = LinRange(1.15, 1.05, nr_s) .* 1000
rd_S = LinRange(20.75, 31.25, nr_s) ./ 10000
ld_S = LinRange(5500., 6800., nr_s) ./ 10000
rmd_S = LinRange(14.4, 10.7, nr_s) .* 1000

max_gna = 30.0
max_gkf_S = ones(nr_s) * 4
max_gks_S = LinRange(16, 25, nr_s)
max_gca_S = LinRange(0.038, 0.029, nr_s)

αm = ones(nr_s) * 22.0
βm = ones(nr_s) * 13.0
αh_S = ones(nr_s) * 0.5
βh_S = ones(nr_s) * 4
αn_S = ones(nr_s) * 1.5
βn_S = ones(nr_s) * 0.1
αq_S = ones(nr_s) * 1.5
βq_S = LinRange(0.025, 0.038, nr_s)
αp_D = 0.008
βp_D = LinRange(0.014, 0.016, nr_s)

Vthca_S = VthCa_newTA[1, :]
Apeak_S = LinRange(0.103, 0.123, nr_s)
tpeak_S = LinRange(110., 100., nr_s)
tetanic_S = LinRange(0.393, 0.491, nr_s)

γh = ones(nr_s)
IF = LinRange(1., 1., nr_s)

EL = 0; ENa = 120; ECa = 140; EK = -10; Ee = 70; Ei = -16
Ge = 0.6; Gi = 0.5
Vd0 = 0.; Vs0 = 0.;
rp = 5.0
pd=1; 

const step = 0.05
_int(num) = floor(Int, num)

MN_list = []
for i in 1:nr_s
    gls = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6
    gld = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6
    gc = ((2/(((70*ld_S[i])/(π*rd_S[i]^2)) + ((70*ls_S[i])/(π*rs_S[i]^2)))))*1e6
    rn = 1 / (gls + ((gld*gc)/(gld+gc)))
    Vth = round(rn * rheobase_S[i], digits=2)
    cs = 2*π*ls_S[i]*rs_S[i]*1*1e3
    cd = 2*π*ld_S[i]*rd_S[i]*1*1e3
    gna = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3
    gkf = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3
    gks = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3
    gca = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3

    S_MN = Dict(:Cs => cs, :Cd => cd, :Irh => rheobase_S[i], :gc => gc,:gNa => gna, :gKs => gks, :gKf => gkf, :gCa => gca,  :gld => gld, :gls => gls, :αp => αp_D, :βp => βp_D[i], :αq => αq_S[i], :βq => βq_S[i], :αm => αm[i], :βm => βm[i],:αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:Vth => Vth, :VthCa => Vthca_S[i], :Rn => rn,:Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh => γh[i],"type" => "All Type")
    push!(MN_list, S_MN)
end
MNs = MN_list

files = sort(filter(f -> occursin(r"(Day\d{2})_trial(\d+)\.npy$", f), readdir(force_dir)))
parsed = map(f -> begin
        m = match(r"(Day\d{2})_trial(\d+)\.npy$", f)
        (day_str = m.captures[1], day = parse(Int, m.captures[1][4:end]), trial = parse(Int, m.captures[2]), path = joinpath(force_dir, f))
    end, files)
sort!(parsed, by = x -> (x.day, x.trial))


for item in parsed
    day_str = item.day_str
    trial = item.trial
    println("\nProcessing $(day_str) trial $(lpad(trial,2,'0'))")

    tr = npzread(item.path)
    sim_length = length(tr)
    interp = LinearInterpolation(1:sim_length, tr)
    resampled_indices = range(1, sim_length, length=Int(round(sim_length/2048))*1000)
    tr = interp(resampled_indices)
    sim_length = length(tr)
    amplitude = maximum(tr)

    global tspan = 0:0.05:sim_length
    sim_time = _int(sim_length / 0.05)

    m=0.0; h=1.0; n=0.0; q=0.0; p=0.0; m0=0.0; h0=1.0; n0=0.0; q0=0.0; p0=0.0
    t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
    global pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s)

    global spike_arr = zeros((nr_s, sim_time))
    global vs_arr = zeros((nr_s, sim_time))
    global pt_arr = zeros((nr_s, sim_time))
    global mt_arr = zeros((nr_s, sim_time))
    global ht_arr = zeros((nr_s, sim_time))
    global nt_arr = zeros((nr_s, sim_time))
    global qt_arr = zeros((nr_s, sim_time))

    global commoninput = tr
    global inh_input = fill(0.00001 * amplitude, sim_length)
    global full_input = zeros((nr_s, sim_length))
    global independantnoise = zeros((nr_s, sim_length))

    @time spike_array, vs_array, pt_arr = run_sim()

    mkpath(joinpath(out_root, day_str))
    output_file = joinpath(out_root, day_str, "DR_S7_$(day_str)_trial$(lpad(trial,2,'0')).jld2")
    JLD2.save(output_file, Dict("spike_array" => spike_array))
    println("Saved simulation results for $(day_str) trial $(lpad(trial,2,'0')) to $output_file")
end



Processing Day01 trial 02
1099.786330 seconds (42.45 G allocations: 692.771 GiB, 8.93% gc time, 0.03% compilation time)
Saved simulation results for Day01 trial 02 to Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S7_alltrials/Day01/DR_S7_Day01_trial02.jld2

Processing Day01 trial 05
709.887885 seconds (41.82 G allocations: 683.383 GiB, 9.68% gc time)
Saved simulation results for Day01 trial 05 to Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S7_alltrials/Day01/DR_S7_Day01_trial05.jld2

Processing Day01 trial 09
669.663728 seconds (42.29 G allocations: 690.481 GiB, 10.03% gc time)
Saved simulation results for Day01 trial 09 to Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S7_alltrials/Day01/DR_S7_Day01_trial09.jld2

Processing Day01 trial 10
664.003768 seconds (43.71 G allocations: 713.536 GiB, 9.59% gc time)
Saved simulation results for Day01 trial 10 to Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S7

## This code is to produce the firing rates for each motoneuron to create the histograms

In [ ]:
# Set sampling rate
sampling_rate = 20000  # Hz

# List of force conditions
force_conditions = [ 1] #, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

# force_conditions = [02, 03, 04, 07,  13]
# force_conditions = [01, 02,03, 04, 05,07,10, 11, 12, 13, 14]
# force_conditions = [01,03,07, 08, 09, 10, 11, 12, 13, 14]

# force_conditions = [01,03,04, 05, 06, 07,08,09,[10, 11, 12, 13, 14]
# force_conditions = [01,02,03,04,05,06,07, 08, 09, 10]

# Create a figure with subplots (4 columns)
n_plots = length(force_conditions)
n_rows = ceil(Int, n_plots / 5)  # Calculate the number of rows based on 4 columns

# Create a plot grid with 4 columns
p = Plots.plot(legend=false, grid=false, xlabel="Firing Rate (Hz)", ylabel="Counts", title="Histogram of Firing Rates", layout=(n_rows, 5), size=(2000, 500))

# Loop through each force condition
for (i, force) in enumerate(force_conditions)
    day_str = lpad(force, 2, '0')  # Format day number with leading zero
    
    # Construct the file path for the current force condition
    force_file = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1/DR_rep_trial_$(day_str)_scaled09.jld2"


    # Load the spike array for the current force condition
    JLD2.@load force_file spike_array

    # Initialize array for storing firing rates for this file
    all_rates = []

    # Calculate rates for each motor unit
    # Initialize array for storing average firing rates (one per motor unit)
    avg_rates_per_neuron = Float64[]

    for mn in 1:size(spike_array, 1)
        spike_pos = findall(x -> x == 1.0, spike_array[mn, :])
        if length(spike_pos) > 1
            ISIs = diff(spike_pos) ./ sampling_rate  # ISIs in seconds
            avg_rate = 1 / mean(ISIs)  # Hz
            if avg_rate <= 80  # 
                push!(avg_rates_per_neuron, avg_rate)
            end
        end
    end

    println(avg_rates_per_neuron)

    # save_path = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/firings/S1/force$(force)_firing_rates_rep_scaled08.jld2"


    # JLD2.@save save_path avg_rates_per_neuron

    if !isempty(avg_rates_per_neuron)
        histogram!(p, avg_rates_per_neuron,bins=20,
            normalize=false,
            xlabel="Firing Rate (Hz)",
            ylabel="Counts",
            title="Force $force",
            subplot=i,
            xlim=[0, 80])
    else
        println("Warning: No valid firing rates for force $force")
    end

end

# Show the plot
display(p)


In [59]:
############## LOAD FORCE → SCALE (0.8/0.9/1.0) → RUN MODEL (ms) → SAVE ONLY BEST ##############

using NPZ, Interpolations, Distributions, Random, JLD2, Printf, Statistics
using JSON, StatsBase, HDF5

# --- bring your simulation functions into scope (must define `run_sim`) ---
folder_path = "./functions_run_simulations/"
for file in readdir(folder_path)
    endswith(file, ".jl") && include(joinpath(folder_path, file))
end
@assert isdefined(Main, :run_sim) "run_sim() is not defined. Make sure files in $(folder_path) define it."

# ===================== USER CONFIG =====================
days        = [1, 3,4,5,6,7,8,9,10,11,12,13,14]
force_root  = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/Scaled_force/Raw/S1"
Fs_force    = 2048.0                 
scales      = [0.8,0.9, 1.1]
out_root    = "Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1"

# Experimental peaks JSON (per Day & Trial: exp["1"]["17"]["peak_hz"])
exp_json    = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Data_final/S1_peak_rate_histogram.json"
exp_peaks   = JSON.parsefile(exp_json)

# Peak selection target: “5 bins to the right” with 1 Hz bins ⇒ 5 Hz
TARGET_DELTA_HZ = 5.0
BINS    = collect(0:1:80)             # 1 Hz bins
CENTERS = (BINS[1:end-1] .+ BINS[2:end]) ./ 2
Fs_spikes = 1000.0                    # spikes on ms grid → 1000 Hz


# ======================================================
# ======================================================
# ---------- STATIC NEURON PROPERTIES + POOL ----------
# ======================================================
# ======================================================

JLD2.@load "Data_MN_Model/Data_final/Subjects_optim_process/VthCa_TA_subject1_gamma1.jld2" VthCa_newTA
nr_s = length(VthCa_newTA)

rheobase_S = LinRange(3.5, 6.5, nr_s) .+ rand(Random.default_rng(), Normal(0.0, 0.05), nr_s)
rs_S  = LinRange(38.75, 41.25, nr_s) ./ 10000
ls_S  = LinRange(77.5, 82.5,  nr_s) ./ 10000
rms_S = LinRange(1.15, 1.05,  nr_s) .* 1000
rd_S  = LinRange(20.75, 31.25, nr_s) ./ 10000
ld_S  = LinRange(5500., 6800., nr_s) ./ 10000
rmd_S = LinRange(14.4, 10.7,  nr_s) .* 1000

max_gna  = 30.0
max_gkf_S = ones(nr_s) .* 4
max_gks_S = LinRange(16, 25, nr_s)
max_gca_S = LinRange(0.038, 0.029, nr_s)

αm = ones(nr_s) .* 22.0
βm = ones(nr_s) .* 13.0
αh_S = ones(nr_s) .* 0.5
βh_S = ones(nr_s) .* 4.0
αn_S = ones(nr_s) .* 1.5
βn_S = ones(nr_s) .* 0.1
αq_S = ones(nr_s) .* 1.5
βq_S = LinRange(0.025, 0.038, nr_s)
αp_D = 0.008
βp_D = LinRange(0.014, 0.016, nr_s)

Vthca_S   = VthCa_newTA[1, :]
Apeak_S   = LinRange(0.103, 0.123, nr_s)
tpeak_S   = LinRange(110., 100., nr_s)
tetanic_S = LinRange(0.393, 0.491, nr_s)

γh = ones(nr_s)
IF = LinRange(1., 1., nr_s)

# Model constants (globals if your code expects them)
EL = 0.0; ENa = 120.0; ECa = 140.0; EK = -10.0; Ee = 70.0; Ei = -16.0
Ge = 0.6; Gi = 0.5
Vd0 = 0.0; Vs0 = 0.0
rp  = 5.0
pd  = 1

# Build MN pool
MN_list = Vector{Dict}(undef, nr_s)
for i in 1:nr_s
    gls = ((2π*rs_S[i]*ls_S[i])/rms_S[i])*1e6
    gld = ((2π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6
    gc  = ((2/(((70*ld_S[i])/(π*rd_S[i]^2)) + ((70*ls_S[i])/(π*rs_S[i]^2)))))*1e6
    rn  = 1 / (gls + (gld*gc)/(gld+gc))
    Vth = round(rn * rheobase_S[i], digits=2)
    cs  = 2π*ls_S[i]*rs_S[i]*1e3
    cd  = 2π*ld_S[i]*rd_S[i]*1e3
    gna = 2π*ls_S[i]*rs_S[i]*max_gna*1e3
    gkf = 2π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3
    gks = 2π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3
    gca = 2π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3

    MN_list[i] = Dict(
        :Cs=>cs, :Cd=>cd, :Irh=>rheobase_S[i], :gc=>gc,
        :gNa=>gna, :gKs=>gks, :gKf=>gkf, :gCa=>gca,
        :gld=>gld, :gls=>gls, :αp=>αp_D, :βp=>βp_D[i], :αq=>αq_S[i], :βq=>βq_S[i],
        :αm=>αm[i], :βm=>βm[i], :αh=>αh_S[i], :βh=>βh_S[i], :αn=>αn_S[i], :βn=>βn_S[i],
        :Vth=>Vth, :VthCa=>Vthca_S[i], :Rn=>rn,
        :Tetanic=>tetanic_S[i], :Apeak=>Apeak_S[i], :IF=>IF[i], :tpeak=>tpeak_S[i], :γh=>γh[i],
        "type"=>"All Type"
    )
end
MNs = MN_list  

# ---------- helpers ----------
function parse_day_trial(path::AbstractString)
    mday = match(r"Day0*(\d+)", path)
    mtr  = match(r"trial0*(\d+)", path)
    day  = isnothing(mday) ? nothing : parse(Int, mday.captures[1])
    tr   = isnothing(mtr)  ? nothing : parse(Int, mtr.captures[1])
    return day, tr
end

function load_force_h5(p::AbstractString, default_Fs::Float64)
    h5open(p, "r") do h5
        force = read(h5["force"])
        attrs = HDF5.attributes(h5)
        Fs = default_Fs
        if haskey(attrs, "Fs")
            Fs_val = read(attrs["Fs"])
            Fs = float(Fs_val isa AbstractArray ? Fs_val[1] : Fs_val)
        end
        return vec(force), Fs
    end
end

function get_exp_peak_hz(exp::Dict, day::Int, trial::Int)
    dnode = haskey(exp, string(day)) ? exp[string(day)] :
            (haskey(exp, @sprintf("%02d", day)) ? exp[@sprintf("%02d", day)] : nothing)
    dnode === nothing && return nothing
    tnode = haskey(dnode, string(trial)) ? dnode[string(trial)] :
            (haskey(dnode, @sprintf("%02d", trial)) ? dnode[@sprintf("%02d", trial)] : nothing)
    tnode === nothing && return nothing
    return haskey(tnode, "peak_hz") ? float(tnode["peak_hz"]) : nothing
end

# avg FRs (Hz) from ms spike raster
function avg_rates_from_spikes(spikes::AbstractArray{<:Real,2}; Fs::Real=Fs_spikes, cap_hz::Real=80.0)
    n_units, T = size(spikes)
    rates = Float64[]
    @inbounds for i in 1:n_units
        idx = findall(!iszero, spikes[i, :])
        if length(idx) > 1
            isi = diff(idx) ./ Fs
            r = 1 / mean(isi)
            (r <= cap_hz) && push!(rates, r)
        end
    end
    return rates
end

# ---------- build file index (day,trial) → path ----------
force_files = Dict{Tuple{Int,Int},String}()
for (root, _dirs, files) in walkdir(force_root)
    for name in files
        if endswith(lowercase(name), ".h5")
            p = joinpath(root, name)
            day, tr = parse_day_trial(p)
            (day !== nothing && tr !== nothing) && (force_files[(day,tr)] = p)
        end
    end
end
isempty(force_files) && error("No .h5 files with DayXX & trialYY found under: $force_root")

by_day = Dict{Int, Vector{Int}}()
for (day, tr) in keys(force_files)
    day in days || continue
    push!(get!(by_day, day, Int[]), tr)
end
for d in keys(by_day)
    unique!(by_day[d]); sort!(by_day[d])
end

# ---------- MAIN: per day, per trial (MS GRID THROUGHOUT) ----------
for day in days


    ###### CHECK TRIALS FOR A GIVEN DAY ######
    ##########################################
    haskey(by_day, day) || (println("\nNo trials for Day$(lpad(day,2,'0')) — skipping."); continue)
    day_str = @sprintf("Day%02d", day)
    trials  = by_day[day]
    println("\n=== $day_str → trials: $(trials) ===")

    day_out = joinpath(out_root, day_str)
    isdir(day_out) || mkpath(day_out)
    
    ###### GET FORCE INPUTS ######
    ##############################

    for tr in trials
        fpath = force_files[(day,tr)]
        force_raw, Fs_in = load_force_h5(fpath, Fs_force)
    

        sim_length = length(force_raw)
        interp = LinearInterpolation(1:sim_length, force_raw)
        resampled_indices = range(1, sim_length, length=Int(round(sim_length/2048))*1000)
        tr_ms = interp(resampled_indices)
        sim_length = length(tr_ms)
        amplitude = maximum(tr_ms)

        # ===========================================================

        exp_peak = get_exp_peak_hz(exp_peaks, day, tr) #### RETUNRS PEAK FIRING OF THE DISTRIBUTION ####
        if exp_peak === nothing
            @warn "No experimental peak for $day_str trial $(lpad(tr,2,'0')); will choose highest simulated peak."
        end

        results = Dict{Float64, Dict{Symbol,Any}}()  # s => (:spike_array, :rates, :sim_peak_hz)

        for s in scales
            tr_scaled = tr_ms .* s

            # Simulation settings
            global tspan = 0:0.05:sim_length
            sim_time = _int(sim_length / 0.05)

            # Initialize state variables
            m=0.0; h=1.0; n=0.0; q=0.0; p=0.0; m0=0.0; h0=1.0; n0=0.0; q0=0.0; p0=0.0
            t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
            global pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s)

            global spike_arr = zeros((nr_s, sim_time))
            global vs_arr = zeros((nr_s, sim_time))
            global pt_arr = zeros((nr_s, sim_time))
            global mt_arr = zeros((nr_s, sim_time))
            global ht_arr = zeros((nr_s, sim_time))
            global nt_arr = zeros((nr_s, sim_time))
            global qt_arr = zeros((nr_s, sim_time))

            global commoninput = tr_scaled
            global inh_input = fill(0.00001 * amplitude, sim_length)
            global full_input = zeros((nr_s, sim_length))
            global independantnoise = zeros((nr_s, sim_length))


            ######################### RUN SIMULATION ##########################
            ####################################################################
            @time spike_array, vs_array, pt_array = run_sim()
            ####################################################################

            #### CALCULATE DISTRIBUTION OF FIRING RATES
            # avg FRs + sim peak (Hz) with Fs=1000
            rates = avg_rates_from_spikes(spike_array; Fs=Fs_spikes, cap_hz=80.0)
            sim_peak_hz = if isempty(rates)
                -Inf
            else
                h = fit(Histogram, rates, BINS)
                isempty(h.weights) ? -Inf : CENTERS[argmax(h.weights)]
            end

            # Save THIS run immediately; keep its path for later (in case it's the best)
            s_tag   = replace(@sprintf("%.1f", s), "." => "")  # e.g., 0.8 -> "08"
            base    = "DR_S1_$(day_str)_trial$(lpad(tr,2,'0'))_s$(s_tag).jld2"
            outfile = joinpath(out_root, day_str, base)
            JLD2.@save outfile spike_array
            println("Saved (scale=$(s)) → $(outfile)")

            # Stash results for selection
            results[s] = Dict(
                :spike_array => spike_array,
                :rates       => rates,
                :sim_peak_hz => sim_peak_hz,
                :path        => outfile
            )



            #### CALCULATE DISTRIBUTION OF FIRING RATES
            # avg FRs + sim peak (Hz) with Fs=1000
            rates = avg_rates_from_spikes(spike_array; Fs=Fs_spikes, cap_hz=80.0)
            sim_peak_hz = if isempty(rates)
                -Inf
            else
                h = fit(Histogram, rates, BINS)
                isempty(h.weights) ? -Inf : CENTERS[argmax(h.weights)]
            end

            results[s] = Dict(:spike_array=>spike_array, :rates=>rates, :sim_peak_hz=>sim_peak_hz)
        end

    # ------ WHICH SCALE IS THE BEST? (robust index-based selection) ------
    # Build vector of simulated peaks (Hz) in the same order as `scales`
    sim_peaks = [results[s][:sim_peak_hz] for s in scales]

    # Helper: pick index for max peak, ignoring -Inf
    function idx_max_peak(peaks::Vector{Float64})
        if all(!isfinite, peaks)
            return nothing
        else
            return argmax(peaks)  # returns Int index
        end
    end

    idx_chosen::Union{Nothing,Int} = nothing

    if exp_peak !== nothing && isfinite(exp_peak)
        target = exp_peak + TARGET_DELTA_HZ
        # score per scale (same order as `scales`)
        scores = [ isfinite(sim_peaks[i]) ?
                abs(sim_peaks[i] - target) + (sim_peaks[i] < exp_peak ? 0.25 : 0.0) :
                Inf
                for i in eachindex(scales) ]

        if all(!isfinite, sim_peaks)
            idx_chosen = idx_max_peak(sim_peaks)   # may be `nothing` -> handled below
        else
            idx_chosen = argmin(scores)            # Int index
        end
    else
        idx_chosen = idx_max_peak(sim_peaks)
    end

    # Fallback: if still nothing (all -Inf), just take the first scale
    if idx_chosen === nothing
        @warn "All simulated peaks were non-finite for $day_str trial $(lpad(tr,2,'0')); defaulting to first scale."
        idx_chosen = 1
    end

    chosen_scale = scales[idx_chosen]
    # ---------------------------------------------------------------------

    # ---- MARK the chosen run as GOOD (keep both original and GOOD) ----
    chosen_path = results[chosen_scale][:path]
    good_path   = replace(chosen_path, ".jld2" => "_GOOD.jld2")
    cp(chosen_path, good_path; force=true)   # use mv(...) if you prefer to move instead of copy
    println("Marked BEST (scale=$(chosen_scale)) → $(good_path)")


end
end



=== Day01 → trials: [1, 6, 9, 11, 14, 17, 21, 24, 27, 30] ===
396.178926 seconds (23.46 G allocations: 390.370 GiB, 12.35% gc time, 0.07% compilation time)
Saved (scale=0.8) → Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1/Day01/DR_S1_Day01_trial01_s08.jld2
536.252148 seconds (23.63 G allocations: 392.903 GiB, 12.51% gc time)
Saved (scale=0.9) → Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1/Day01/DR_S1_Day01_trial01_s09.jld2
567.997898 seconds (24.47 G allocations: 405.509 GiB, 11.77% gc time)
Saved (scale=1.1) → Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_S1/Day01/DR_S1_Day01_trial01_s11.jld2


KeyError: KeyError: key :path not found